# English → Gujarati: Kaggle runner

All the logic lives in the `en_gu` package of the repo (`src/en_gu/`); this notebook only installs it and runs the
pipeline (prepare → tokenizer → train → evaluate → export). Hyperparameters are in `configs/base.yaml`; override
them in the **Settings** cell.

**Kaggle setup:** accelerator GPU T4 ×2 (P100 also works), Internet on, and the dataset
`english-to-gujarati-machine-translation-dataset` added as input. Use **Save Version → Save & Run All (Commit)** so
training runs in the background.

**Outputs:** `/kaggle/working/best.pt` is the self-contained model for the app (download it and put it next to
`app.py`). While training, `/kaggle/working/artifacts/train/best.pt` is refreshed whenever validation improves, so
you can grab an intermediate model.

If the repo is private, add it to Kaggle as a dataset instead of cloning it.

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/AnshBhavsar1654/en-gu-transformer.git"
if not os.path.isdir("en-gu-transformer"):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
os.chdir("en-gu-transformer")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[train]"], check=True)
subprocess.run(["git", "log", "-1", "--oneline"], check=True)  # which code version this run used

## Settings

In [ ]:
import glob

found = glob.glob("/kaggle/input/**/train.en", recursive=True)
CORPUS_DIR = os.path.dirname(found[0]) if found else "en-gu"
OUT = "/kaggle/working"

OVERRIDES = [
    f"paths.corpus_dir={CORPUS_DIR}",
    f"paths.processed_dir={OUT}/data/processed",
    f"paths.eval_dir={OUT}/data/eval",
    f"paths.artifacts_dir={OUT}/artifacts",
    # Change hyperparameters here, e.g.:
    # "data.max_sentences=null",            # whole corpus (~3.07M pairs) instead of the first 1M
    # "train.max_epochs=30",
    # "train.resume_checkpoint=/kaggle/input/<previous-run>/artifacts/train/last.pt",
]
ARGS = " ".join(f"--set {o}" for o in OVERRIDES)
print(ARGS)

## Run the pipeline

In [ ]:
!python -m en_gu all {ARGS}

## Results

In [ ]:
import json
import shutil

shutil.copy(f"{OUT}/artifacts/best.pt", f"{OUT}/best.pt")
print("Saved", f"{OUT}/best.pt", f"({os.path.getsize(f'{OUT}/best.pt') / 1e6:.0f} MB)")
print(json.dumps(json.load(open(f"{OUT}/artifacts/eval/metrics.json")), indent=2))

In [ ]:
import matplotlib.pyplot as plt

history = json.load(open(f"{OUT}/artifacts/train/history.json"))["history"]
if history:
    steps = [h["step"] for h in history]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(steps, [h["val_loss"] for h in history], marker="o")
    ax1.set(title="Validation loss", xlabel="update")
    ax2.plot(steps, [h["val_bleu"] for h in history], marker="o", label="BLEU")
    ax2.plot(steps, [h["val_chrf"] for h in history], marker="o", label="chrF++")
    ax2.set(title="Greedy scores on validation subset", xlabel="update")
    ax2.legend()
    plt.tight_layout()
    plt.show()